In [1]:
import numpy as np
import pandas as pd

In [2]:
train_df= pd.read_csv('PubMed_200k_RCT/train.csv')
dev_df = pd.read_csv('PubMed_200k_RCT/dev.csv')
test_df = pd.read_csv('PubMed_200k_RCT/test.csv')
train_df.head()

,abstract_id,line_id,abstract_text,line_number,total_lines,target
0,24491034,24491034_0_11,The emergence of HIV as a chronic condition me...,0,11,BACKGROUND
1,24491034,24491034_1_11,This paper describes the design and evaluation...,1,11,BACKGROUND
2,24491034,24491034_2_11,This study is designed as a randomised control...,2,11,METHODS
3,24491034,24491034_3_11,The intervention group will participate in the...,3,11,METHODS
4,24491034,24491034_4_11,The program is based on self-efficacy theory a...,4,11,METHODS


In [3]:
train_df = train_df.dropna(subset=['abstract_text'])
print(train_df['abstract_text'].isna().sum())
print(dev_df['abstract_text'].isna().sum())
print(test_df['abstract_text'].isna().sum())
print(train_df.describe())
print(train_df.info())

0
0
0
        abstract_id   line_number   total_lines
count  2.211860e+06  2.211860e+06  2.211860e+06
mean   1.777430e+07  5.709641e+00  1.241928e+01
std    5.372625e+06  4.054368e+00  3.317234e+00
min    1.279170e+06  0.000000e+00  3.000000e+00
25%    1.282092e+07  2.000000e+00  1.000000e+01
50%    1.850567e+07  5.000000e+00  1.200000e+01
75%    2.234881e+07  8.000000e+00  1.400000e+01
max    2.652916e+07  5.000000e+01  5.100000e+01
<class 'pandas.core.frame.DataFrame'>
Index: 2211860 entries, 0 to 2211860
Data columns (total 6 columns):
 #   Column         Dtype 
---  ------         ----- 
 0   abstract_id    int64 
 1   line_id        object
 2   abstract_text  object
 3   line_number    int64 
 4   total_lines    int64 
 5   target         object
dtypes: int64(3), object(3)
memory usage: 118.1+ MB
None


In [4]:
print(dev_df.describe())
print(dev_df.info())

        abstract_id   line_number   total_lines
count  2.893200e+04  28932.000000  28932.000000
mean   1.781278e+07      5.676172     12.352343
std    5.356084e+06      3.999521      3.176805
min    1.336526e+06      0.000000      4.000000
25%    1.287862e+07      2.000000     10.000000
50%    1.855817e+07      5.000000     12.000000
75%    2.239816e+07      8.000000     14.000000
max    2.643621e+07     27.000000     28.000000
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28932 entries, 0 to 28931
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   abstract_id    28932 non-null  int64 
 1   line_id        28932 non-null  object
 2   abstract_text  28932 non-null  object
 3   line_number    28932 non-null  int64 
 4   total_lines    28932 non-null  int64 
 5   target         28932 non-null  object
dtypes: int64(3), object(3)
memory usage: 1.3+ MB
None


In [5]:
print(test_df.describe())
print(test_df.info())

        abstract_id   line_number   total_lines
count  2.949300e+04  29493.000000  29493.000000
mean   1.773442e+07      5.805344     12.610687
std    5.390987e+06      4.093325      3.279820
min    1.334248e+06      0.000000      4.000000
25%    1.273415e+07      2.000000     10.000000
50%    1.832493e+07      5.000000     12.000000
75%    2.236804e+07      9.000000     14.000000
max    2.642272e+07     27.000000     28.000000
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29493 entries, 0 to 29492
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   abstract_id    29493 non-null  int64 
 1   line_id        29493 non-null  object
 2   abstract_text  29493 non-null  object
 3   line_number    29493 non-null  int64 
 4   total_lines    29493 non-null  int64 
 5   target         29493 non-null  object
dtypes: int64(3), object(3)
memory usage: 1.4+ MB
None


In [6]:
# # Drop the unneccassry column
train_df = train_df.drop(columns=['abstract_id', 'line_id'])
dev_df = dev_df.drop(columns=['abstract_id', 'line_id'])
# med_data_test = med_data_test.drop(columns=['abstract_id', 'line_id'])
train_df.head()

,abstract_text,line_number,total_lines,target
0,The emergence of HIV as a chronic condition me...,0,11,BACKGROUND
1,This paper describes the design and evaluation...,1,11,BACKGROUND
2,This study is designed as a randomised control...,2,11,METHODS
3,The intervention group will participate in the...,3,11,METHODS
4,The program is based on self-efficacy theory a...,4,11,METHODS


In [7]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

2026-02-24 13:47:59.391160: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-24 13:48:04.154811: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-24 13:48:08.435295: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [8]:
train_sentences = train_df["abstract_text"].tolist()
train_labels = train_df["target"].tolist()

val_sentences = dev_df["abstract_text"].tolist()
val_labels = dev_df["target"].tolist()

In [9]:
print(len(train_sentences))
print(len(train_labels))
print(len(val_sentences))
print(len(val_labels))

2211860
2211860
28932
28932


## Step 3: Encode Labels

In [10]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

label_encoder = LabelEncoder()
train_labels_encoded = label_encoder.fit_transform(train_labels)
val_labels_encoded = label_encoder.transform(val_labels)

train_labels_onehot = to_categorical(train_labels_encoded)
val_labels_onehot = to_categorical(val_labels_encoded)

## Step 4: Tokenization

In [11]:
max_vocab = 2000000
tokenizer = Tokenizer(num_words=max_vocab, oov_token="<OOV>")
tokenizer.fit_on_texts(train_sentences)

train_sequences = tokenizer.texts_to_sequences(train_sentences)
val_sequences = tokenizer.texts_to_sequences(val_sentences)

## Step 5: Padding


In [12]:
max_length = 55
train_padded = pad_sequences(train_sequences, maxlen= max_length, padding= "post")
val_padded = pad_sequences(val_sequences, maxlen= max_length, padding = "post")

## Build The LSTM Model
### Simple LSTM Architecture

In [13]:
model = tf.keras.Sequential([layers.Embedding(input_dim = max_vocab, output_dim = 128, input_length = max_length),
                             layers.LSTM(128),
                             layers.Dense(64, activation = "relu"),
                             layers.Dense(5, activation = "softmax")])

/home/cse23160/Downloads/Cryptomedbench/tf_env/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
I0000 00:00:1771921135.300283    5109 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6127 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070, pci bus id: 0000:01:00.0, compute capability: 8.6


In [14]:
## Comile Model
model.compile(loss="categorical_crossentropy",
              optimizer="adam",
              metrics = ["accuracy"])

In [15]:
# Training
history = model.fit(train_padded, train_labels_onehot,
                    validation_data=(val_padded, val_labels_onehot),
                    epochs = 5,
                    batch_size = 32)

Epoch 1/5


/home/cse23160/Downloads/Cryptomedbench/tf_env/lib/python3.10/site-packages/tensorflow/python/framework/indexed_slices.py:446: UserWarning: Converting sparse IndexedSlices to a dense Tensor with 256000000 elements. This may consume a large amount of memory.
  warnings.warn(
2026-02-24 13:49:02.176844: W external/local_xla/xla/service/gpu/llvm_gpu_backend/default/nvptx_libdevice_path.cc:41] Can't find libdevice directory ${CUDA_DIR}/nvvm/libdevice. This may result in compilation or runtime failures, if the program we try to run uses routines from libdevice.
Searched for CUDA in the following directories:
  ./cuda_sdk_lib
  ipykernel_launcher.runfiles/cuda_nvcc
  ipykernel_launcher.runfiles/cuda_nvdisasm
  ipykernel_launcher.runfiles/nvidia_nvshmem
  ipykern/cuda_nvcc
  ipykern/cuda_nvdisasm
  ipykern/nvidia_nvshmem
  
  /usr/local/cuda
  /opt/cuda
  /home/cse23160/Downloads/Cryptomedbench/tf_env/lib/python3.10/site-packages/tensorflow/python/platform/../../../nvidia/cuda_nvcc
  /home/cs

UnknownError: Graph execution error:

Detected at node sequential/lstm/lstm_cell/recurrent_kernel/Initializer/Sign defined at (most recent call last):
  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main

  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code

  File "/home/cse23160/Downloads/Cryptomedbench/tf_env/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/home/cse23160/Downloads/Cryptomedbench/tf_env/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/home/cse23160/Downloads/Cryptomedbench/tf_env/lib/python3.10/site-packages/ipykernel/kernelapp.py", line 758, in start

  File "/home/cse23160/Downloads/Cryptomedbench/tf_env/lib/python3.10/site-packages/tornado/platform/asyncio.py", line 211, in start

  File "/usr/lib/python3.10/asyncio/base_events.py", line 603, in run_forever

  File "/usr/lib/python3.10/asyncio/base_events.py", line 1909, in _run_once

  File "/usr/li

In [ ]:
# Get Prediction probabilitties
val_pred_probs = model.predict(val_padded)

In [ ]:
# Convert Predicted Probabilites to class index
val_pred_classes = np.argmax(val_pred_probs, axis = 1)

# Convert Truel labels from one-hot 
val_true_classes = np.argmax(val_labels_onehot, axis =1)

In [ ]:
## Classification Report
from sklearn.metrics import classification_report
print(classification_report(val_true_classes,
                            val_pred_classes,
                            target_names = label_encoder.classes_))

In [ ]:
# Confusion Matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(val_true_classes, val_pred_classes)
print(cm)

In [ ]:
#visualize the Confusion Matrix
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot = True, fmt="d", xticklabels = label_encoder.classes_,
            yticklabels = label_encoder.classes_, cmap = "Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix-LSTM")
plt.show()

In [ ]:
## Class wise accuracy
class_accuracy = cm.diagonal()/cm.sum(axis=1)
for label, acc in zip(label_encoder.classes_, class_accuracy):
    print(f"{label}: {acc:3f}")

In [ ]:
# Overall Accuracy
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(val_true_classes, val_pred_classes)
print("Accuracy:", accuracy)

In [ ]:
## Research-Level Evalauation 
from sklearn.metrics import f1_score

macro_f1 = f1_score(val_true_classes, val_pred_classes, average="macro")
weighted_f1 = f1_score(val_true_classes, val_pred_classes, average="weighted")

print("Macro F1:", macro_f1)
print("Weighted F1:", weighted_f1)

## Using Bidirectional Model

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers

In [ ]:
train_sentences = train_df["abstract_text"].astype(str).values
val_sentences   = dev_df["abstract_text"].astype(str).values
test_sentences  = test_df["abstract_text"].astype(str).values

train_labels = train_df["target"].values
val_labels   = dev_df["target"].values
test_labels  = test_df["target"].values

In [ ]:
label_encoder = LabelEncoder()

train_labels_encoded = label_encoder.fit_transform(train_labels)
val_labels_encoded   = label_encoder.transform(val_labels)
test_labels_encoded  = label_encoder.transform(test_labels)

print(label_encoder.classes_)

In [ ]:
max_vocab = 20000

tokenizer = Tokenizer(num_words=max_vocab, oov_token="<OOV>")
tokenizer.fit_on_texts(train_sentences)

train_sequences = tokenizer.texts_to_sequences(train_sentences)
val_sequences   = tokenizer.texts_to_sequences(val_sentences)
test_sequences  = tokenizer.texts_to_sequences(test_sentences)

In [ ]:
max_length = 55

train_padded = pad_sequences(train_sequences, maxlen=max_length, padding="post")
val_padded   = pad_sequences(val_sequences, maxlen=max_length, padding="post")
test_padded  = pad_sequences(test_sequences, maxlen=max_length, padding="post")

In [ ]:
train_padded = np.array(train_padded, dtype=np.int32)
val_padded   = np.array(val_padded, dtype=np.int32)
test_padded  = np.array(test_padded, dtype=np.int32)

train_labels_encoded = np.array(train_labels_encoded, dtype=np.int32)
val_labels_encoded   = np.array(val_labels_encoded, dtype=np.int32)
test_labels_encoded  = np.array(test_labels_encoded, dtype=np.int32)

In [ ]:
model = tf.keras.Sequential([
    
    layers.Embedding(
        input_dim=max_vocab,
        output_dim=128,
        input_length=max_length
    ),
    
    layers.Bidirectional(layers.LSTM(128, return_sequences=True)),
    layers.Bidirectional(layers.LSTM(64)),

    
    layers.Dropout(0.5),
    
    layers.Dense(64, activation="relu"),
    
    layers.Dense(5, activation="softmax")
])


In [ ]:
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    train_padded,
    train_labels_encoded,
    validation_data=(val_padded, val_labels_encoded),
    epochs=5,
    batch_size=32
)

In [ ]:
# Get Prediction probabilitties
val_pred_probs = model.predict(val_padded)

In [ ]:
# Convert Predicted Probabilites to class index
val_pred_classes = np.argmax(val_pred_probs, axis = 1)

# Convert Truel labels from one-hot 
val_true_classes = np.argmax(val_labels_onehot, axis =1)

In [ ]:
## Classification Report
from sklearn.metrics import classification_report
print(classification_report(val_true_classes,
                            val_pred_classes,
                            target_names = label_encoder.classes_))

In [ ]:
# Confusion Matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(val_true_classes, val_pred_classes)
print(cm)

In [ ]:
#visualize the Confusion Matrix
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot = True, fmt="d", xticklabels = label_encoder.classes_,
            yticklabels = label_encoder.classes_, cmap = "Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix-LSTM")
plt.show()

In [ ]:
## Class wise accuracy
class_accuracy = cm.diagonal()/cm.sum(axis=1)
for label, acc in zip(label_encoder.classes_, class_accuracy):
    print(f"{label}: {acc:3f}")

In [ ]:
# Overall Accuracy
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(val_true_classes, val_pred_classes)
print("Accuracy:", accuracy)

In [ ]:
## Research-Level Evalauation 
from sklearn.metrics import f1_score

macro_f1 = f1_score(val_true_classes, val_pred_classes, average="macro")
weighted_f1 = f1_score(val_true_classes, val_pred_classes, average="weighted")

print("Macro F1:", macro_f1)
print("Weighted F1:", weighted_f1)